# CLEIDS-Edge — Notebook 03: Training CLEIDS-Edge

Trains the hybrid CNN-LSTM architecture (`build_cleids_edge`, Notebook 02) on the four ready benchmark datasets (NSL-KDD, CICIDS2017, UNSW-NB15, TON_IoT) for both Binary (benign/attack) and Multi-class classification tasks.

**GPU runtime (L4) is required for this notebook.** Cell 4 enforces GPU visibility and halts execution if no GPU is available.

**This notebook cannot be executed by Claude Code** — there is no tool available to run cells against a remote Colab kernel, and the local machine has no GPU. Every cell is written to be run **by you**, in Antigravity, connected to a live Colab GPU session. No metric here is real until you run it.

**Data bridge**: Notebook 01 ran locally, so `data/processed/*.npz` (772MB total, largest file 439MB — over GitHub's file-size limit) exists only on the local machine, not in Colab's `/content`. Cell 2 downloads it from a local HTTP server tunneled via localtunnel — **this tunnel is only alive while that Claude Code session's background processes are running**; if Cell 2 fails with a connection error, ask for it to be restarted, or re-run Notebook 01 directly in Colab instead.

Independent per-dataset sections (same convention as Notebook 01) so IoT-23 can be appended later without touching the other four. See `CLEIDS_PROJECT_BRIEF.md` for overall thesis pipeline context.

## 1. Repo setup (clone/pull + auth)

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/NehlTech/CLEIDS-Edge.git"
REPO_DIR = "/content/CLEIDS-Edge"

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise RuntimeError("GITHUB_TOKEN not found. Add it to Colab secrets (key icon, left sidebar).")

AUTH_REMOTE = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", AUTH_REMOTE, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", AUTH_REMOTE], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

## 2. Data bridge — download processed splits from the local machine

Pulls `data/processed/<dataset>/{train,val,test}.npz` for the four ready datasets from a localtunnel-exposed HTTP server on the machine that ran Notebook 01 locally. If this fails with a connection error, the tunnel has gone down.

In [ ]:
import urllib.request

BRIDGE_URL = "https://thin-bars-matter.loca.lt"  # update if the tunnel was restarted
DATASETS = ["nsl-kdd", "cicids2017", "unsw-nb15", "ton-iot"]

def fetch(url, dest):
    req = urllib.request.Request(url, headers={"bypass-tunnel-reminder": "true"})
    with urllib.request.urlopen(req) as resp, open(dest, "wb") as f:
        f.write(resp.read())

for name in DATASETS:
    os.makedirs(f"data/processed/{name}", exist_ok=True)
    for split in ["train", "val", "test"]:
        dest = f"data/processed/{name}/{split}.npz"
        if os.path.exists(dest):
            print(f"[skip] {dest} already present")
            continue
        print(f"Downloading {name}/{split}.npz ...")
        fetch(f"{BRIDGE_URL}/{name}/{split}.npz", dest)
        print(f"  -> {os.path.getsize(dest)/1e6:.1f} MB")
    for extra in ["label_classes.json", "feature_names.json"]:
        dest = f"data/processed/{name}/{extra}"
        if not os.path.exists(dest):
            fetch(f"{BRIDGE_URL}/{name}/{extra}", dest)

if not os.path.exists("data/processed/preprocessing_manifest.json"):
    fetch(f"{BRIDGE_URL}/preprocessing_manifest.json", "data/processed/preprocessing_manifest.json")
print("\nAll processed data downloaded.")

## 3. Google Drive mount (backup for model checkpoints/results/figures)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/CLEIDS_Edge"
DRIVE_MODELS = os.path.join(DRIVE_ROOT, "models")
DRIVE_RESULTS = os.path.join(DRIVE_ROOT, "results")
DRIVE_FIGURES = os.path.join(DRIVE_ROOT, "figures")
for d in (DRIVE_MODELS, DRIVE_RESULTS, DRIVE_FIGURES):
    os.makedirs(d, exist_ok=True)
print("Drive ready at:", DRIVE_ROOT, "-- backup target in case hours of GPU compute finish before a successful git push.")

## 4. Setup & GPU Verification

In [ ]:
import sys
import time
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

tf.random.set_seed(42)
np.random.seed(42)

# GPU Enforcement -- this notebook must not silently fall back to CPU
gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError(
        "CRITICAL ERROR: No GPU runtime detected! Notebook 03 requires a GPU (L4). "
        "Please switch your runtime to GPU in Colab (Runtime -> Change runtime type -> GPU) "
        "before running this notebook."
    )
print(f"[HARDWARE OK] GPU visible: {gpus}")
print("TensorFlow version:", tf.__version__)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from models import build_cleids_edge

for d in ["models", "results", "figures"]:
    os.makedirs(d, exist_ok=True)

with open("data/processed/preprocessing_manifest.json") as f:
    prep_manifest = json.load(f)
print("Loaded preprocessing manifest. Ready datasets:", list(prep_manifest["datasets"].keys()))

## 5. Evaluation & Plotting Utilities

`validate_against_manifest` loads `{train,val,test}.npz` and asserts row counts match Notebook 01's `preprocessing_manifest.json` exactly — **stops with a clear error on any mismatch**, never proceeds silently. `train_and_evaluate_model` builds via `build_cleids_edge`, trains with the specified callbacks, evaluates once on test, and flags a warning if accuracy lands at/near the majority-class baseline (non-convergence is reported, never hidden). AUC failures are reported as `None`, never silently replaced with a placeholder value.

In [ ]:
def validate_against_manifest(dataset_name, train_data, val_data, test_data):
    expected = prep_manifest["datasets"][dataset_name]["shapes"]
    actual = {"train": train_data["X_cnn"].shape[0], "val": val_data["X_cnn"].shape[0], "test": test_data["X_cnn"].shape[0]}
    for split, actual_n in actual.items():
        expected_n = expected[split][0]
        if actual_n != expected_n:
            raise RuntimeError(
                f"[{dataset_name}] {split} row count mismatch: Notebook 01 manifest says {expected_n:,}, "
                f"loaded {actual_n:,}. Stopping rather than proceeding on mismatched data."
            )
    print(f"[VALIDATE] {dataset_name}: train={actual['train']:,} val={actual['val']:,} test={actual['test']:,} "
          f"-- matches Notebook 01 manifest exactly")


def calculate_fpr(y_true, y_pred, binary=True):
    """Calculate False Positive Rate (FPR = FP / (FP + TN)); macro-averaged one-vs-rest for multiclass."""
    cm = confusion_matrix(y_true, y_pred)
    if binary:
        tn, fp, fn, tp = cm.ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    else:
        fprs = []
        for i in range(len(cm)):
            tp = cm[i, i]
            fp = cm[:, i].sum() - tp
            fn = cm[i, :].sum() - tp
            tn = cm.sum() - (tp + fp + fn)
            class_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
            fprs.append(class_fpr)
        fpr = float(np.mean(fprs))
    return float(fpr)


def plot_confusion_matrix(y_true, y_pred, class_names, save_path, title):
    """Plot and save 300 DPI confusion matrix figure."""
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    plt.figure(figsize=(8, 6) if len(class_names) <= 10 else (12, 10), dpi=300)
    sns.heatmap(
        cm, annot=(len(class_names) <= 15), fmt="d", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names
    )
    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved confusion matrix plot to {save_path}")


def plot_training_curves(history, save_path, title):
    """Plot and save 300 DPI training and validation loss/accuracy curves."""
    plt.figure(figsize=(12, 5), dpi=300)
    plt.subplot(1, 2, 1)
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Val Loss")
    plt.title(f"{title} - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)

    plt.subplot(1, 2, 2)
    acc_key = "accuracy" if "accuracy" in history.history else "acc"
    val_acc_key = "val_" + acc_key
    plt.plot(history.history[acc_key], label="Train Accuracy")
    plt.plot(history.history[val_acc_key], label="Val Accuracy")
    plt.title(f"{title} - Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved training curves to {save_path}")

In [ ]:
def train_and_evaluate_model(dataset_name, binary=True, batch_size=256):
    """Train CLEIDS-Edge on given dataset split and evaluate on held-out test set."""
    task_str = "binary" if binary else "multiclass"
    print("\n" + "=" * 70)
    print(f"[TRAINING] {dataset_name} ({task_str.upper()})")
    print("=" * 70)

    data_dir = os.path.join("data/processed", dataset_name)
    train_data = np.load(os.path.join(data_dir, "train.npz"))
    val_data = np.load(os.path.join(data_dir, "val.npz"))
    test_data = np.load(os.path.join(data_dir, "test.npz"))

    validate_against_manifest(dataset_name, train_data, val_data, test_data)

    X_train, X_val, X_test = train_data["X_cnn"], val_data["X_cnn"], test_data["X_cnn"]

    with open(os.path.join(data_dir, "label_classes.json")) as f:
        label_info = json.load(f)
    class_names = label_info["classes"]
    num_classes = len(class_names)
    input_dim = X_train.shape[1]

    if binary:
        y_train = train_data["y_bin"].astype(np.float32)
        y_val = val_data["y_bin"].astype(np.float32)
        y_test = test_data["y_bin"].astype(np.float32)
    else:
        y_train_idx = train_data["y_multi"].astype(np.int32)
        y_val_idx = val_data["y_multi"].astype(np.int32)
        y_test_idx = test_data["y_multi"].astype(np.int32)
        y_train = tf.keras.utils.to_categorical(y_train_idx, num_classes=num_classes)
        y_val = tf.keras.utils.to_categorical(y_val_idx, num_classes=num_classes)
        y_test = tf.keras.utils.to_categorical(y_test_idx, num_classes=num_classes)

    model = build_cleids_edge(input_dim=input_dim, num_classes=num_classes, binary=binary)
    ckpt_path = f"models/cleids_edge_{dataset_name}_{task_str}.keras"

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
        tf.keras.callbacks.ModelCheckpoint(filepath=ckpt_path, monitor="val_loss", save_best_only=True, verbose=0),
    ]

    start_time = time.time()
    try:
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=50,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=1,
        )
    except tf.errors.ResourceExhaustedError:
        print(f"[OOM RECOVERY] OOM with batch_size={batch_size}. Retrying with batch_size=128...")
        batch_size = 128
        model = build_cleids_edge(input_dim=input_dim, num_classes=num_classes, binary=binary)
        start_time = time.time()
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=50,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=1,
        )

    train_time_sec = float(time.time() - start_time)
    epochs_run = len(history.history["loss"])
    early_stopped = epochs_run < 50
    print(f"Training complete in {train_time_sec:.2f}s ({train_time_sec/60:.1f} min), "
          f"{epochs_run}/50 epochs, batch_size_used={batch_size}, "
          f"{'early-stopped' if early_stopped else 'ran full 50 epochs'}.")

    if os.path.exists(ckpt_path):
        model = tf.keras.models.load_model(ckpt_path)

    y_pred_raw = model.predict(X_test, batch_size=512, verbose=0)
    if binary:
        y_pred_prob = y_pred_raw.ravel()
        y_pred_cls = (y_pred_prob >= 0.5).astype(int)
        y_true_cls = y_test.astype(int)
        try:
            auc = float(roc_auc_score(y_true_cls, y_pred_prob))
        except ValueError as e:
            auc = None
            print(f"[WARN] AUC could not be computed: {e}")
        acc = float(accuracy_score(y_true_cls, y_pred_cls))
        prec = float(precision_score(y_true_cls, y_pred_cls, zero_division=0))
        rec = float(recall_score(y_true_cls, y_pred_cls, zero_division=0))
        f1 = float(f1_score(y_true_cls, y_pred_cls, zero_division=0))
        fpr = calculate_fpr(y_true_cls, y_pred_cls, binary=True)
        plot_cls_names = ["Benign", "Attack"]
        per_class_report = None
    else:
        y_pred_cls = np.argmax(y_pred_raw, axis=1)
        y_true_cls = y_test_idx
        try:
            auc = float(roc_auc_score(y_test, y_pred_raw, multi_class="ovr", average="macro"))
        except ValueError as e:
            auc = None
            print(f"[WARN] Multiclass AUC could not be computed: {e}")
        acc = float(accuracy_score(y_true_cls, y_pred_cls))
        prec = float(precision_score(y_true_cls, y_pred_cls, average="macro", zero_division=0))
        rec = float(recall_score(y_true_cls, y_pred_cls, average="macro", zero_division=0))
        f1 = float(f1_score(y_true_cls, y_pred_cls, average="macro", zero_division=0))
        fpr = calculate_fpr(y_true_cls, y_pred_cls, binary=False)
        plot_cls_names = class_names
        # labels=range(num_classes) is required: without it, sklearn infers the label set from
        # whatever appears in y_true/y_pred, which can be fewer than num_classes (e.g. NSL-KDD's
        # ultra-rare spy/perl/phf may be entirely absent from the test set) -- that mismatch
        # against a fixed-length target_names raises ValueError.
        per_class_report = classification_report(
            y_true_cls, y_pred_cls, labels=list(range(num_classes)),
            target_names=class_names, output_dict=True, zero_division=0,
        )
        print(f"\n[{dataset_name} MULTI-CLASS PER-CLASS METRICS]")
        print(pd.DataFrame(per_class_report).transpose().to_string())

    fig_cm_path = f"figures/confusion_matrix_{dataset_name}_{task_str}.png"
    fig_curve_path = f"figures/training_curve_{dataset_name}_{task_str}.png"
    plot_confusion_matrix(y_true_cls, y_pred_cls, plot_cls_names, fig_cm_path, f"CLEIDS-Edge {dataset_name} ({task_str.upper()})")
    plot_training_curves(history, fig_curve_path, f"CLEIDS-Edge {dataset_name} ({task_str.upper()})")

    metrics = {
        "accuracy": acc, "precision": prec, "recall": rec, "f1": f1,
        "auc": auc, "fpr": fpr, "train_time_sec": round(train_time_sec, 2),
        "epochs_run": epochs_run, "batch_size_used": batch_size, "checkpoint_path": ckpt_path,
    }
    if per_class_report is not None:
        metrics["per_class_report"] = per_class_report

    auc_str = f"{auc:.4f}" if auc is not None else "N/A"
    print(f"[{dataset_name} {task_str.upper()} RESULTS] Acc={acc:.4f} Prec={prec:.4f} Rec={rec:.4f} "
          f"F1={f1:.4f} AUC={auc_str} FPR={fpr:.4f}")

    majority_baseline = np.bincount(y_true_cls.astype(int)).max() / len(y_true_cls)
    if acc <= majority_baseline + 0.01:
        print(f"[WARNING] Accuracy ({acc:.4f}) is at/near the majority-class baseline "
              f"({majority_baseline:.4f}) -- this model may not have learned anything beyond "
              f"predicting the majority class. Reporting as-is, not hiding it.")

    return metrics

## 6. NSL-KDD Training

`num_classes=40` includes NSL-KDD's 17 test-only novel attack classes (a documented dataset property, not a bug) — expect near-zero recall specifically on those in the per-class table, not full-dataset non-convergence.

**After this section, read the time-estimate cell below before continuing** — per the run request, total time for all 8 runs should be estimated here before committing to the rest.

In [ ]:
t0 = time.time()
nsl_bin_metrics = train_and_evaluate_model("nsl-kdd", binary=True)
nsl_mc_metrics = train_and_evaluate_model("nsl-kdd", binary=False)
nsl_elapsed = time.time() - t0

print(f"\n[ESTIMATE] nsl-kdd (binary+multiclass) took {nsl_elapsed/60:.1f} min total.")
print(f"[ESTIMATE] Rough projection for the remaining 3 datasets x 2 tasks: "
      f"~{nsl_elapsed/60*3:.0f}-{nsl_elapsed/60*5:.0f} min more (varies with dataset size).")
print("[ESTIMATE] STOP AND REVIEW: NSL-KDD's train set (1,212,169 rows) is the second-smallest of "
      "the four post-SMOTE; CICIDS2017 (2,275,692 rows) and TON_IoT (2,022,420 rows) are roughly "
      "double, so the real total will likely run higher than this naive projection for those two. "
      "If the projected total exceeds a few hours, decide now whether to let Colab run unattended "
      "(e.g. overnight) or step in, per the run instructions.")

## 7. CICIDS2017 Training

Largest training set of the four (2,275,692 rows post-capped-SMOTE) -- expect this to be the slowest.

In [ ]:
cic_bin_metrics = train_and_evaluate_model("cicids2017", binary=True)
cic_mc_metrics = train_and_evaluate_model("cicids2017", binary=False)

## 8. UNSW-NB15 Training

Smallest training set of the four (504,000 rows) -- expect this to be the fastest.

In [ ]:
unsw_bin_metrics = train_and_evaluate_model("unsw-nb15", binary=True)
unsw_mc_metrics = train_and_evaluate_model("unsw-nb15", binary=False)

## 9. TON_IoT Training

Second-largest training set (2,022,420 rows post-SMOTE).

In [ ]:
ton_bin_metrics = train_and_evaluate_model("ton-iot", binary=True)
ton_mc_metrics = train_and_evaluate_model("ton-iot", binary=False)

## IoT-23 (pending)

**Not yet processed/trained.** IoT-23 has no `data/processed/iot-23/` splits yet (Notebook 01's IoT-23 section is still pending). Once ready, this section will call `train_and_evaluate_model("iot-23", binary=...)` following the exact same pattern as the four sections above — no changes needed there.

This placeholder is intentional — do not treat IoT-23 as silently skipped or forgotten.

In [ ]:
print("IoT-23: pending -- see markdown cell above. Not trained in this run.")

## 10. Consolidated Results & Artifact Saving

In [ ]:
main_results = {
    "nsl-kdd": {"binary": nsl_bin_metrics, "multiclass": nsl_mc_metrics},
    "cicids2017": {"binary": cic_bin_metrics, "multiclass": cic_mc_metrics},
    "unsw-nb15": {"binary": unsw_bin_metrics, "multiclass": unsw_mc_metrics},
    "ton-iot": {"binary": ton_bin_metrics, "multiclass": ton_mc_metrics},
}

results_path = "results/main_results.json"
with open(results_path, "w") as f:
    json.dump(main_results, f, indent=2)
print(f"Wrote consolidated results to {results_path}")

## 11. Backup to Drive + push to GitHub

In [ ]:
# Drive backup -- models/ included, not just results/figures, since these represent hours of
# GPU compute that would otherwise be lost if the Colab session ends before a successful push.
shutil.copy2(results_path, os.path.join(DRIVE_RESULTS, "main_results.json"))
for fn in os.listdir("figures"):
    shutil.copy2(os.path.join("figures", fn), os.path.join(DRIVE_FIGURES, fn))
for fn in os.listdir("models"):
    shutil.copy2(os.path.join("models", fn), os.path.join(DRIVE_MODELS, fn))
print("Drive backup of models/, figures/, results/ complete at", DRIVE_ROOT)

# All checkpoint files are tiny for this architecture (~483KB each, ~4MB for 8 models --
# param count is independent of input_dim, see Notebook 02), so plain git works with no LFS
# needed -- verified below rather than assumed.
large_files = [os.path.join("models", fn) for fn in os.listdir("models")
               if os.path.getsize(os.path.join("models", fn)) > 50 * 1024 * 1024]
if large_files:
    print(f"[WARN] {len(large_files)} checkpoint file(s) exceed 50MB -- excluded from git push, "
          f"Drive backup above is the only copy: {large_files}")
else:
    print("All checkpoint files are under 50MB -- safe to push to git directly.")

model_files = [f"models/{fn}" for fn in os.listdir("models") if f"models/{fn}" not in large_files]
figure_files = [f"figures/{fn}" for fn in os.listdir("figures")]
add_paths = ["notebooks/03_Training_CLEIDS_Edge.ipynb", results_path] + model_files + figure_files

subprocess.run(["git", "-C", REPO_DIR, "add"] + add_paths, check=True)
commit_res = subprocess.run(
    ["git", "-C", REPO_DIR, "commit", "-m", "Notebook 03: train CLEIDS-Edge on 4 datasets (binary + multiclass)"],
    capture_output=True, text=True,
)
print(commit_res.stdout, commit_res.stderr)
if commit_res.returncode == 0:
    subprocess.run(["git", "-C", REPO_DIR, "push", "origin", "HEAD"], check=True)
    print("Pushed to GitHub.")
else:
    print("Nothing new to commit (or commit failed) -- see output above.")

## 12. Final Consolidated Summary

Paste this cell's output back for review before Notebook 04 (baselines).

In [ ]:
print("\n" + "=" * 95)
print("CLEIDS-Edge -- Notebook 03 Headline Results Summary")
print("=" * 95)
header = (f"{'Dataset':<12} | {'Task':<10} | {'Accuracy':<8} | {'Precision':<9} | {'Recall':<8} | "
          f"{'F1-Score':<8} | {'AUC':<7} | {'FPR':<7} | {'Time (s)':<8} | {'Epochs':<6}")
print(header)
print("-" * 95)
for ds_name, tasks in main_results.items():
    for task_name, m in tasks.items():
        auc_str = f"{m['auc']:.4f}" if m["auc"] is not None else "N/A"
        row = (f"{ds_name:<12} | {task_name:<10} | {m['accuracy']:<8.4f} | {m['precision']:<9.4f} | "
               f"{m['recall']:<8.4f} | {m['f1']:<8.4f} | {auc_str:<7} | {m['fpr']:<7.4f} | "
               f"{m['train_time_sec']:<8.1f} | {m['epochs_run']:<6}")
        print(row)
print("=" * 95)
print("\nIoT-23: pending -- will be appended as an independent section once Notebook 01's IoT-23 data is ready.")
print("Full details (per-class reports, checkpoint paths) are in results/main_results.json.")